In [ ]:
# ===============================================================
# CSIRO Image2Biomass – Training (Weighted R² Validation)
# ===============================================================
import os, gc, cv2, numpy as np, pandas as pd
from tqdm import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.model_selection import KFold
from torch.amp import GradScaler, autocast
import wandb
from datetime import datetime
from sklearn.model_selection import GroupKFold
import random
# ---------------------------------------------------------------
# 1. CONFIG (memory-safe + R² metric)
# ---------------------------------------------------------------
class CFG:
    BASE_PATH       = 'csiro-biomass'
    TRAIN_CSV       = os.path.join(BASE_PATH, 'train.csv')
    TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, 'train')
    MODEL_DIR       = 'out'
    N_FOLDS         = 5

    MODEL_NAME = 'eva02_small_patch14_336.mim_in22k_ft_in1k'      # safe & matches inference
    IMG_SIZE   = 336                  # fits easily
    PRETRAINED = True
    FREEZE_BACKBONE = False

    BATCH_SIZE   = 2
    GRAD_ACC     = 8                  # effective batch = 8
    NUM_WORKERS  = 1
    EPOCHS       = 100
    LR           = 1e-4
    WD           = 0.01 #1e-2 convnext
    PATIENCE     = 10
    WARMUP_EPOCHS = 2

    DETERMINISTIC = True
    SEED = 122

    TARGET_COLS    = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']
    DERIVED_COLS   = ['Dry_Clover_g', 'Dry_Dead_g']
    ALL_TARGET_COLS = ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']
    R2_WEIGHTS     = np.array([0.1, 0.1, 0.1, 0.2, 0.5])  # matches metric
    W_SPECIES = 0.25
    W_STATE = 0.25
    W_CONT = 0.5

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device : {CFG.DEVICE}")
print(f"Backbone: {CFG.MODEL_NAME} | Size: {CFG.IMG_SIZE}")

def set_seed(seed=42, deterministic=True):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ---------------------------------------------------------------
# 2. WEIGHTED R² METRIC (your function)
# ---------------------------------------------------------------
def global_weighted_r2_score(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Computes the globally weighted R² score as described in the evaluation.
    
    y_true, y_pred: shape (N, 5)
    weights: [0.1, 0.1, 0.1, 0.2, 0.5] (from CFG)
    """
    weights_matrix = np.tile(CFG.R2_WEIGHTS, (y_true.shape[0], 1))
    # y_bar_w = (sum(w_j * y_j)) / (sum(w_j))
    weighted_sum = np.sum(weights_matrix * y_true)
    total_weight = np.sum(weights_matrix)
    y_bar_w = weighted_sum / total_weight # This is a single scalar value
    # SS_res = sum(w_j * (y_j - y_pred_j)^2)
    ss_res = np.sum(weights_matrix * (y_true - y_pred) ** 2)
    # SS_tot = sum(w_j * (y_j - y_bar_w)^2)
    ss_tot = np.sum(weights_matrix * (y_true - y_bar_w) ** 2)
    # R²_w = 1 - (SS_res / SS_tot)
    r2_w = 1 - (ss_res / ss_tot)
    return r2_w
def weighted_r2_score(y_true: np.ndarray, y_pred: np.ndarray):
    """
    y_true, y_pred: shape (N, 5)
    weights: [0.1, 0.1, 0.1, 0.2, 0.5]
    """
    weights = CFG.R2_WEIGHTS
    r2_scores = []
    for i in range(5):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        ss_res = np.sum((y_t - y_p) ** 2)
        ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    r2_scores = np.array(r2_scores)
    weighted_r2 = np.sum(r2_scores * weights) / np.sum(weights)
    return weighted_r2
# ---------------------------------------------------------------
# 3. AUGMENTATIONS
# ---------------------------------------------------------------
def get_spatial_transforms():
    # These will be applied to BOTH images identically
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        # A.PadIfNeeded(
        #     min_height=CFG.IMG_SIZE, # Set this to 1024
        #     min_width=CFG.IMG_SIZE,  # Set this to 1024
        #     border_mode=cv2.BORDER_REFLECT_101 # Best padding mode
        # ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
    ], 
    p=1.0,
    additional_targets={'image_right': 'image'},
    seed=CFG.SEED 
    )
def get_photometric_transforms():
    # These will be applied INDEPENDENTLY to each half
    return A.Compose([
        A.ColorJitter(brightness=0.5,contrast=0.5,saturation=0.5,hue=0.0,p=0.5),
        # A.GaussianBlur(blur_limit=(0, 2), p=0.3),
        # A.GaussNoise(std_range=(0,0.1),mean_range=(0,0),p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0, seed=CFG.SEED)

def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=(-10, 10), p=0.3,
                 interpolation=cv2.INTER_LINEAR,
                 border_mode=cv2.BORDER_REFLECT_101),
        A.ColorJitter(brightness=0.1, contrast=0.1, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0, seed=CFG.SEED)

def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        # A.PadIfNeeded(
        #     min_height=CFG.IMG_SIZE, # Set this to 1024
        #     min_width=CFG.IMG_SIZE,  # Set this to 1024
        #     border_mode=cv2.BORDER_REFLECT_101 # Best padding mode
        # ),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0)
# ---------------------------------------------------------------
# X. Discriminative learning rate
# ---------------------------------------------------------------
def create_optimizer_groups(model, lr_heads, lr_backbone):
    """
    Creates parameter groups for the optimizer with differential learning rates.
    """
    # 1. Identify the parameter groups
    # This automatically handles the 'module.' prefix from nn.DataParallel
    prefix = 'module.' if isinstance(model, nn.DataParallel) else ''
    
    backbone_params = [p for n, p in model.named_parameters() 
                       if n.startswith(f'{prefix}backbone.') and p.requires_grad]
                       
    head_params     = [p for n, p in model.named_parameters() 
                       if n.startswith(f'{prefix}head_') and p.requires_grad] # 'head_' matches all 3
    
    # 2. Create parameter "layer groups"
    layer_groups = [
        ('Backbone', lr_backbone, backbone_params),
        ('Heads',    lr_heads,    head_params)
    ]

    # placeholder
    parameters = []

    # store params & learning rates
    # print("--- Optimizer Parameter Groups ---")
    for idx, (name, lr, params) in enumerate(layer_groups):
        
        # Check if the parameter group is empty
        num_params = sum(p.numel() for p in params)
        if len(params) == 0:
            print(f"Warning: Group '{name}' has 0 parameters.")
            continue # Don't add an empty group
        
        # display info
        # print(f'{idx}: lr = {lr:.6f}, group = {name} ({len(params)} tensors, {num_params} params)')

        # append layer parameters
        parameters.append({
            'params': params,
            'lr':     lr
        })
        
    return parameters
# ---------------------------------------------------------------
# 4. DATASET
# ---------------------------------------------------------------
class BiomassDataset(Dataset):
    def __init__(self, df, transform, photometric_transform, img_dir):
        self.df        = df
        self.transform = transform
        self.ph_transform = photometric_transform
        self.img_dir   = img_dir
        self.paths     = df['image_path'].values
        self.labels    = df[CFG.ALL_TARGET_COLS].values.astype(np.float32)
        self.aux_cat_labels = df[CFG.CAT_AUX_COLS].values.astype(np.int64)
        self.aux_cont_labels = df[CFG.CONT_AUX_COLS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img  = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        mid = w // 2
        left  = img[:, :mid]
        right = img[:, mid:]
        if self.transform:
            transformed = self.transform(image=left, image_right=right)
            left  = transformed['image']
            right = transformed['image_right']

        # 2. Apply PHOTOMETRIC transforms (independently)
        left  = self.ph_transform(image=left)['image']
        right = self.ph_transform(image=right)['image']

        label = torch.from_numpy(self.labels[idx])
        aux_cat_label   = torch.from_numpy(self.aux_cat_labels[idx])
        aux_cont_label  = torch.from_numpy(self.aux_cont_labels[idx])
        return left, right, label, aux_cat_label, aux_cont_label

# ---------------------------------------------------------------
# 5. MODEL (safe pretrained loading)
# ---------------------------------------------------------------
class BiomassModel(nn.Module):
    def __init__(self, model_name, n_species, n_states, n_cont_targets, pretrained=True, freeze_backbone=False):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=0, global_pool='avg')
        nf = self.backbone.num_features
        comb = nf * 2

        def head():
            return nn.Sequential(
                nn.Linear(comb, comb // 2),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(comb // 2, 1)
            )
        self.head_total = head()
        self.head_gdm   = head()
        self.head_green = head()

        # --- ADD AUXILIARY HEADS ---
        # A shared "neck" for auxiliary tasks
        # self.aux_neck = nn.Sequential(
        #     nn.Linear(comb, comb // 2),
        #     nn.ReLU(inplace=True),
        #     nn.Dropout(0.3)
        # )
        
        # 1. Categorical Heads (predict logits)
        # self.head_species = nn.Linear(comb // 2, n_species)
        # self.head_state   = nn.Linear(comb // 2, n_states)
        
        # 2. Continuous Heads (predict scaled values)
        # self.head_cont = nn.Linear(comb // 2, n_cont_targets) # Predicts all 4 at once
        # ---

        if pretrained:
            self.load_pretrained()
        if freeze_backbone:
            self.freeze_backbone()

    def load_pretrained(self):
        try:
            state_dict = timm.create_model(CFG.MODEL_NAME, pretrained=True, num_classes=0).state_dict()
            self.backbone.load_state_dict(state_dict, strict=False)
            print("Pretrained weights loaded (CPU)")
        except Exception as e:
            print(f"Warning: Pretrained load failed: {e}")

    def freeze_backbone(self):
        """
        Freezes all parameters in the backbone.
        """
        print("Freezing backbone parameters.")
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, left, right):
        fl = self.backbone(left)
        fr = self.backbone(right)
        x  = torch.cat([fl, fr], dim=1)

        # Get main biomass predictions
        p_total = self.head_total(x)
        p_gdm   = self.head_gdm(x)
        p_green = self.head_green(x)
        
        # --- Get Auxiliary Predictions ---
        # x_aux = self.aux_neck(x) # Pass features through aux_neck first
        
        # p_species = self.head_species(x_aux) # (batch_size, n_species)
        # p_state   = self.head_state(x_aux)   # (batch_size, n_states)
        # p_cont    = self.head_cont(x_aux)    # (batch_size, n_cont_targets)
        # ---

        # return (p_total, p_gdm, p_green, p_species, p_state, p_cont)
        return (p_total, p_gdm, p_green)

# ---------------------------------------------------------------
# 6. LOSS (MSE on all 5)
# ---------------------------------------------------------------
def weighted_biomass_loss(p_total, p_gdm, p_green, labels):
    """
    Calculates the 5 individual MSE losses and returns their
    weighted sum, perfectly aligning with the R2 metric weights.
    """
    mse = nn.MSELoss()
    
    # 1. Calculate the 5 individual MSE losses
    loss_total = mse(p_total.squeeze(), labels[:, 4]) # Corresponds to Dry_Total_g
    loss_gdm   = mse(p_gdm.squeeze(),   labels[:, 3]) # Corresponds to GDM_g
    loss_green = mse(p_green.squeeze(), labels[:, 0]) # Corresponds to Dry_Green_g

    # Calculate derived predictions
    p_clover = torch.clamp(p_gdm - p_green, min=0)
    p_dead   = torch.clamp(p_total - p_gdm, min=0)

    loss_clover = mse(p_clover.squeeze(), labels[:, 2]) # Corresponds to Dry_Clover_g
    loss_dead   = mse(p_dead.squeeze(),   labels[:, 1]) # Corresponds to Dry_Dead_g

    # 2. Get the weights
    weights = CFG.R2_WEIGHTS
    
    # 3. Apply the weights to their corresponding losses
    weighted_loss_sum = (
        loss_green  * weights[0] +
        loss_dead   * weights[1] +
        loss_clover * weights[2] +
        loss_gdm    * weights[3] +
        loss_total  * weights[4]
    )
    
    return weighted_loss_sum

# ---------------------------------------------------------------
# 7. VALIDATION WITH WEIGHTED R²
# ---------------------------------------------------------------
@torch.no_grad()
def valid_epoch(model, loader, device):
    model.eval()
    running_loss = 0.0
    preds = {'total':[], 'gdm':[], 'green':[]}
    all_labels = []

    for l, r, lab, aux_cat, aux_cont in tqdm(loader, desc='valid', leave=False):
        l, r, lab = l.to(device, non_blocking=True), r.to(device, non_blocking=True), lab.to(device, non_blocking=True)

        with autocast('cuda',dtype=torch.bfloat16):
        # p_tot, p_gdm, p_green, _, _, _ = model(l, r) # We can ignore aux outputs
            p_tot, p_gdm, p_green = model(l, r) # We can ignore aux outputs
            loss = weighted_biomass_loss(p_tot, p_gdm, p_green, lab)

        running_loss += loss.item() * l.size(0)

        preds['total'].extend(p_tot.cpu().float().numpy().ravel())
        preds['gdm'].extend(p_gdm.cpu().float().numpy().ravel())
        preds['green'].extend(p_green.cpu().float().numpy().ravel())
        all_labels.extend(lab.cpu().float().numpy())

    # Convert to numpy
    pred_total = np.array(preds['total'])
    pred_gdm   = np.array(preds['gdm'])
    pred_green = np.array(preds['green'])
    true_labels = np.stack(all_labels)  # (N, 5)

    # Compute derived
    pred_clover = np.clip(pred_gdm - pred_green, 0, None)
    pred_dead   = np.clip(pred_total - pred_gdm, 0, None)

    # Stack predictions in correct order
    pred_all = np.stack([
        pred_green,      # Dry_Green_g
        pred_dead,       # Dry_Dead_g
        pred_clover,     # Dry_Clover_g
        pred_gdm,        # GDM_g
        pred_total       # Dry_Total_g
    ], axis=1)

    # Compute weighted R²
    weighted_r2 = global_weighted_r2_score(true_labels, pred_all)

    return running_loss / len(loader.dataset), weighted_r2, pred_all, true_labels

# ---------------------------------------------------------------
# 8. TRAINING LOOP
# ---------------------------------------------------------------
loss_fn_categorical = nn.CrossEntropyLoss()
loss_fn_continuous = nn.MSELoss()        # MSE or L1Loss
def train_epoch(model, loader, opt, scheduler, device, scaler):
    model.train()
    running = 0.0

    opt.zero_grad()
    for i, (l, r, lab, aux_cat, aux_cont) in enumerate(tqdm(loader, desc='train', leave=False)):
        l, r, lab = l.to(device, non_blocking=True), r.to(device, non_blocking=True), lab.to(device, non_blocking=True)
        aux_cat = aux_cat.to(device, non_blocking=True)
        aux_cont = aux_cont.to(device, non_blocking=True)
        with autocast('cuda',dtype=torch.bfloat16):
        # p_tot, p_gdm, p_green, p_species, p_state, p_cont = model(l, r)
            p_tot, p_gdm, p_green = model(l, r)
            # 1. Main loss
            loss_reg = weighted_biomass_loss(p_tot, p_gdm, p_green, lab)

        # 2. Auxiliary losses
        # loss_species = loss_fn_categorical(p_species, aux_cat[:, 1]) # Species is 2nd col
        # loss_state   = loss_fn_categorical(p_state,   aux_cat[:, 0]) # State is 1st col
        # loss_cont    = loss_fn_continuous(p_cont, aux_cont)

        # 3. Combine all losses with weights
        # total_loss = (loss_reg + 
        #             CFG.W_SPECIES * loss_species + 
        #             CFG.W_STATE * loss_state + 
        #             CFG.W_CONT * loss_cont)
        total_loss = loss_reg
        
        loss = total_loss / CFG.GRAD_ACC
        scaler.scale(loss).backward()
        # loss.backward()
        running += loss.item() * l.size(0) * CFG.GRAD_ACC

        if (i + 1) % CFG.GRAD_ACC == 0 or (i + 1) == len(loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            # opt.step()
            opt.zero_grad()

    scheduler.step()
    return running / len(loader.dataset)

/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
Backbone: eva02_small_patch14_336.mim_in22k_ft_in1k | Size: 336


In [ ]:


# ---------------------------------------------------------------
# 9. MAIN – 5-FOLD WITH R² TRACKING
# ---------------------------------------------------------------
set_seed(CFG.SEED, CFG.DETERMINISTIC)
print("Loading data...")
df_long = pd.read_csv(CFG.TRAIN_CSV)
df_wide = df_long.pivot(index='image_path', columns='target_name', values='target').reset_index()
df_wide = df_wide[['image_path'] + CFG.ALL_TARGET_COLS]
print(f"{len(df_wide)} training images")

# Aux task
aux_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
df_aux = df_long[aux_cols].drop_duplicates().reset_index(drop=True)

df_wide = df_wide.merge(df_aux, on='image_path', how='left')

df_wide['State_idx'],   STATE_MAP   = pd.factorize(df_wide['State'])
df_wide['Species_idx'], SPECIES_MAP = pd.factorize(df_wide['Species'])

CFG.N_STATES   = len(STATE_MAP)
CFG.N_SPECIES  = len(SPECIES_MAP)
print(f"Found {CFG.N_STATES} states and {CFG.N_SPECIES} species.")

# 2. Convert Date to cyclical features (we'll predict these)
df_wide['Sampling_Date'] = pd.to_datetime(df_wide['Sampling_Date'])
df_wide['day_of_year'] = df_wide['Sampling_Date'].dt.dayofyear
df_wide['day_sin'] = np.sin(2 * np.pi * df_wide['day_of_year'] / 365.25)
df_wide['day_cos'] = np.cos(2 * np.pi * df_wide['day_of_year'] / 365.25)

# 3. Define all continuous columns to be scaled
# We'll predict NDVI and Height, so they must be scaled as targets
CFG.CONT_AUX_COLS = ['Pre_GSHH_NDVI', 'Height_Ave_cm', 'day_sin', 'day_cos']
CFG.CAT_AUX_COLS  = ['State_idx', 'Species_idx']

group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

config = {k: v for k, v in vars(CFG).items() if not k.startswith('_')}
df_wide['group'] = df_wide['State'].astype(str) + "_" + df_wide['Sampling_Date'].astype(str)
kfold = GroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

# Store the best R2 score from each fold
all_fold_best_scores = []

for fold, (tr_idx, val_idx) in enumerate(kfold.split(df_wide,groups=df_wide['group'])):
    print('\n' + '='*70)
    print(f'   FOLD {fold+1}/{CFG.N_FOLDS}   |   {len(tr_idx)} train / {len(val_idx)} val')
    print('='*70)
    wandb.init(
            project="csiro-biomass",
            group=group_name,           # Group all folds under this name
            name=f"{group_name}_f{fold}", # Unique name for this fold
            config=config,              # Log config for each run
            reinit=True                 # Allow re-initialization
        )
    torch.cuda.empty_cache()
    gc.collect()

    tr_df  = df_wide.iloc[tr_idx].reset_index(drop=True)
    val_df = df_wide.iloc[val_idx].reset_index(drop=True)

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()

    # Fit on train, transform both
    tr_df[CFG.CONT_AUX_COLS] = scaler.fit_transform(tr_df[CFG.CONT_AUX_COLS])
    val_df[CFG.CONT_AUX_COLS] = scaler.transform(val_df[CFG.CONT_AUX_COLS])

    tr_set = BiomassDataset(tr_df, get_spatial_transforms(),get_photometric_transforms(), CFG.TRAIN_IMAGE_DIR)
    val_set= BiomassDataset(val_df, None,get_val_transforms(),   CFG.TRAIN_IMAGE_DIR)

    def seed_worker(worker_id):
        s = torch.initial_seed() % 2**32
        np.random.seed(s)
        random.seed(s)
    g = torch.Generator()
    g.manual_seed(CFG.SEED)

    tr_loader  = DataLoader(tr_set,  batch_size=CFG.BATCH_SIZE, shuffle=True,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True, worker_init_fn=seed_worker,generator=g)
    val_loader = DataLoader(val_set, batch_size=CFG.BATCH_SIZE, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker,generator=g)

    print("Building model...")
    model = BiomassModel(
            CFG.MODEL_NAME, 
            n_species=CFG.N_SPECIES,  # Pass in class numbers
            n_states=CFG.N_STATES,
            n_cont_targets=len(CFG.CONT_AUX_COLS), # Pass in number of cont. targets
            pretrained=CFG.PRETRAINED, 
            freeze_backbone=CFG.FREEZE_BACKBONE
        )
    model = model.to(CFG.DEVICE)
    # model = nn.DataParallel(model)

    if CFG.FREEZE_BACKBONE:
        parameters = filter(lambda p: p.requires_grad, model.parameters())
    else:
        parameters = create_optimizer_groups(
            model=model,
            lr_heads=CFG.LR,
            lr_backbone=CFG.LR# fixed for the moment
        )

    optimizer = optim.AdamW(parameters, lr=CFG.LR, weight_decay=CFG.WD, )

    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1e-6, # Start from a very small LR
        end_factor=1.0,
        total_iters=CFG.WARMUP_EPOCHS
    )

    main_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG.EPOCHS - CFG.WARMUP_EPOCHS
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, main_scheduler],
        milestones=[CFG.WARMUP_EPOCHS]
    )
    best_r2 = -np.inf
    patience = 0
    try:
        # wandb.watch(model, log="all", log_freq=100)
        scaler = GradScaler()
        for epoch in range(1, CFG.EPOCHS+1):
            tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, CFG.DEVICE, scaler)
            val_loss, val_r2, _, _ = valid_epoch(model, val_loader, CFG.DEVICE)

            print(f'Epoch {epoch:02d} | '
                    f'TrainLoss {tr_loss:.5f} | '
                    f'ValLoss {val_loss:.5f} | '
                    f'ValR² {val_r2:.4f} {"(BEST)" if val_r2 > best_r2 else ""}')

            if val_r2 > best_r2:
                best_r2 = val_r2
                save_path = os.path.join(CFG.MODEL_DIR, f'best_model_fold{fold}.pth')
                torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(), save_path)
                print(f'   → SAVED (R²: {best_r2:.4f})')
                patience = 0
            else:
                patience += 1
                if patience >= CFG.PATIENCE:
                    print(f'   → EARLY STOP (no improvement in {CFG.PATIENCE} epochs)')
                    break
            log_data = {
                    "train_loss": tr_loss,
                    "val_loss": val_loss,
                    "val_r2": val_r2,
                    "best_r2":best_r2,
                }
                
            wandb.log(log_data, step=epoch)
        
        all_fold_best_scores.append(best_r2)
    finally:
        wandb.finish()
        
# Cleanup
del model, tr_loader, val_loader, optimizer, scheduler
torch.cuda.empty_cache()
gc.collect()
mean_cv_score = np.mean(all_fold_best_scores)
std_cv_score  = np.std(all_fold_best_scores)

print('\n' + '='*70)
print('         FINAL CROSS-VALIDATION SCORE')
print('='*70)
print(f'Public LB Score: 0.58')
print(f'Local CV Score: {mean_cv_score:.3f} ± {std_cv_score:.3f}')
print('\nIndividual Fold Scores:')
for i, score in enumerate(all_fold_best_scores):
    print(f'  Fold {i+1}: {score:.4f}')

import csv

run_message = "differential learning rate, default augs"

log_file = os.path.join(CFG.MODEL_DIR, 'experiment_log.csv')
log_data = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'cv_mean': f"{mean_cv_score:.5f}",
    'cv_std': f"{std_cv_score:.5f}",
    'message': run_message,
    'model': CFG.MODEL_NAME,
    'lr': CFG.LR,
    'epochs': CFG.EPOCHS,
    'batch_size': CFG.BATCH_SIZE,
    'img_size': CFG.IMG_SIZE,
    'frozen': CFG.FREEZE_BACKBONE
}
fieldnames = list(log_data.keys())

# 3. WRITE TO THE CSV FILE
# Check if file exists to write header only once
file_exists = os.path.isfile(log_file)

try:
    with open(log_file, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # Write header if new file
        writer.writerow(log_data) # Append new run data
    
    print(f"\nExperiment results saved to {log_file}")
except Exception as e:
    print(f"\nError saving experiment log: {e}")

    print('\nTraining complete! Best models saved in:', CFG.MODEL_DIR)
    print('Use these in inference with:')
    print('   MODEL_NAME = "convnext_tiny"')
    print('   IMG_SIZE = 512')
# Epoch 01 | TrainLoss 1286.64364 | ValLoss 1206.68745 | ValR² -0.1106 (BEST)
#    → SAVED (R²: -0.1106)

Loading data...
357 training images
Found 4 states and 15 species.

   FOLD 1/5   |   289 train / 68 val


wandb: Currently logged in as: butnaruteodor (butnaruteodor-universitatea-tehnic-gheorghe-asachi-din-ia-i) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Building model...
Pretrained weights loaded (CPU)


Epoch 01 | TrainLoss 1886.39207 | ValLoss 2241.85732 | ValR² -1.0633 (BEST)
   → SAVED (R²: -1.0633)


Epoch 02 | TrainLoss 1773.19937 | ValLoss 1966.07447 | ValR² -0.8095 (BEST)
   → SAVED (R²: -0.8095)


...

Epoch 23 | TrainLoss 166.83882 | ValLoss 191.90902 | ValR² 0.5858 


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 24 | TrainLoss 164.27026 | ValLoss 197.54521 | ValR² 0.5736 
   → EARLY STOP (no improvement in 10 epochs)


best_r2,▁▂▅▇▇▇▇████████████████
train_loss,██▆▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_r2,▁▂▅▇▇▇▇████████████████
best_r2,0.59183
train_loss,166.83882
val_loss,191.90902
val_r2,0.58577



         FINAL CROSS-VALIDATION SCORE
Public LB Score: 0.58
Local CV Score: 0.635 ± 0.051

Individual Fold Scores:
  Fold 1: 0.6800
  Fold 2: 0.5553
  Fold 3: 0.6675
  Fold 4: 0.6787
  Fold 5: 0.5918

Experiment results saved to out/experiment_log.csv
